# Simple 19-Feature Pipeline: Logistic Regression & Calibrated Linear Classifiers
Multi-target intraoperative prediction across **19 Instantaneous Features** with **STRIDE = 10**:
- Models: L2-penalized Logistic Regression, ElasticNet, and Platt Calibrated Probability Models.
- Scalers: Reused from `models/scalers/scaler_<target>.json`
- Output Models: Saved in `models/logistic_regression/`


In [ ]:
import os
import gc
import glob
import json
import random
import warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay
)
import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Environment and Scikit-Learn ML Libraries Loaded Successfully.")


In [ ]:
# ======================================================
# Dataset Loading & 19-Feature Extraction Pipeline (STRIDE = 10)
# ======================================================

USE_FULL_DATASET = True
FORCE_RECOMPUTE_SCALER = False
STRIDE = 10
WINDOW_SIZE = 60

base_dir = os.getcwd()
candidates = [
    os.path.join(base_dir, "patient_labeled_data"),
    os.path.join(base_dir, "..", "patient_labeled_data"),
    os.path.join(base_dir, "..", "..", "patient_labeled_data")
]
input_dir = next((c for c in candidates if os.path.exists(c)), candidates[1])
csv_files = sorted(glob.glob(os.path.join(input_dir, "patient_*_1hz.csv")))

# 19 Raw & Biomarker Features in exact CSV Column Order
base_features = [
    "Solar8000/HR",
    "Solar8000/ART_SBP",
    "Solar8000/ART_DBP",
    "Solar8000/ART_MBP",
    "Solar8000/PLETH_SPO2",
    "Solar8000/RR_CO2",
    "Solar8000/ETCO2",
    "Primus/FIO2",
    "Solar8000/BT"
]

engineered_features = [
    "Feature_Pulse_Pressure",
    "Feature_Shock_Index",
    "Feature_Modified_Shock_Index",
    "Feature_Rate_Pressure_Product",
    "Feature_HR_Mean_60s",
    "Feature_HR_Std_60s",
    "Feature_HR_Delta_60s",
    "Feature_MBP_Mean_60s",
    "Feature_MBP_Std_60s",
    "Feature_MBP_Delta_60s"
]

features_19 = base_features + engineered_features

train_val_files, test_files = train_test_split(csv_files, test_size=0.20, random_state=42, shuffle=True)
train_files, val_files = train_test_split(train_val_files, test_size=0.125, random_state=42, shuffle=True)

train_subset = train_files if USE_FULL_DATASET else train_files[:300]
val_subset   = val_files   if USE_FULL_DATASET else val_files[:50]
test_subset  = test_files  if USE_FULL_DATASET else test_files[:100]

print(f"Dataset Path : {input_dir}")
print(f"Total Patients: {len(csv_files)} | Training: {len(train_subset)} | Val: {len(val_subset)} | Test: {len(test_subset)}")
print(f"Feature Count : {len(features_19)} Features (Instantaneous Biomarkers, STRIDE = {STRIDE})")
for idx, col in enumerate(features_19):
    print(f"  [{idx:02d}] {col}")

def build_dataset_matrix_19(file_list, target_col, oversample_factor=2, stride=STRIDE):
    X_list, y_list = [], []
    req_cols = features_19 + [target_col]
    for file in file_list:
        try:
            df = pd.read_csv(file, usecols=req_cols, dtype=np.float32, engine="c")
            if df.empty or len(df) <= WINDOW_SIZE or target_col not in df.columns:
                continue
            df = df.ffill().bfill().fillna(0)
            arr = df[features_19].to_numpy(dtype=np.float32)
            y_vals = df[target_col].to_numpy(dtype=np.float32)
            n_rows = len(arr)
            indices = np.arange(0, n_rows - 1, stride)
            for i in indices:
                label = y_vals[i]
                if np.isnan(label): continue
                feat_vec = arr[i]
                X_list.append(feat_vec)
                y_list.append(label)
                if label == 1.0 and oversample_factor > 1:
                    for _ in range(oversample_factor - 1):
                        X_list.append(feat_vec)
                        y_list.append(label)
        except Exception:
            continue
    if not X_list:
        return np.empty((0, len(features_19)), dtype=np.float32), np.empty((0,), dtype=np.float32)
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)
def get_optimal_tau(y_t, y_p):
    best_tau, best_f1 = 0.5, -1.0
    for tau in np.linspace(0.01, 0.99, 99):
        f1 = f1_score(y_t, (y_p >= tau).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_tau = f1, tau
    return best_tau

def compute_metrics(y_t, y_p, tau):
    y_b = (y_p >= tau).astype(int)
    auroc = roc_auc_score(y_t, y_p)
    auprc = average_precision_score(y_t, y_p)
    acc = accuracy_score(y_t, y_b)
    bal_acc = balanced_accuracy_score(y_t, y_b)
    prec = precision_score(y_t, y_b, zero_division=0)
    rec = recall_score(y_t, y_b, zero_division=0)
    f1 = f1_score(y_t, y_b, zero_division=0)
    mcc = matthews_corrcoef(y_t, y_b)
    tn, fp, fn, tp = confusion_matrix(y_t, y_b).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return dict(auroc=auroc, auprc=auprc, acc=acc, bal_acc=bal_acc, prec=prec, rec=rec, spec=spec, f1=f1, mcc=mcc, tn=tn, fp=fp, fn=fn, tp=tp)

def print_metrics_table(target_name, model_name, y_te, test_probs, optimal_tau):
    m_def = compute_metrics(y_te, test_probs, 0.50)
    m_opt = compute_metrics(y_te, test_probs, optimal_tau)
    print("=" * 76)
    print(f"  TEST METRICS: {model_name} | TARGET: {target_name}")
    print("=" * 76)
    print(f"Metric                 Default (tau=0.50)       OPTIMAL (tau*={optimal_tau:.2f})")
    print("-" * 76)
    print(f"AUROC (ROC AUC)         : {m_def['auroc']:.4f}                  {m_opt['auroc']:.4f}")
    print(f"AUPRC (PR AUC)          : {m_def['auprc']:.4f}                  {m_opt['auprc']:.4f}")
    print(f"Accuracy                : {m_def['acc']:.4f}                  {m_opt['acc']:.4f}")
    print(f"Balanced Accuracy       : {m_def['bal_acc']:.4f}                  {m_opt['bal_acc']:.4f}")
    print(f"Sensitivity / Recall    : {m_def['rec']:.4f}                  {m_opt['rec']:.4f}")
    print(f"Specificity (TNR)       : {m_def['spec']:.4f}                  {m_opt['spec']:.4f}")
    print(f"Precision (PPV)         : {m_def['prec']:.4f}                  {m_opt['prec']:.4f}")
    print(f"F1 Score                : {m_def['f1']:.4f}                  {m_opt['f1']:.4f}")
    print(f"MCC                     : {m_def['mcc']:.4f}                  {m_opt['mcc']:.4f}")
    print("-" * 76)
    print(f"Confusion Matrix (0.50) : TN={m_def['tn']}, FP={m_def['fp']}, FN={m_def['fn']}, TP={m_def['tp']}")
    print(f"Confusion Matrix (tau*) : TN={m_opt['tn']}, FP={m_opt['fp']}, FN={m_opt['fn']}, TP={m_opt['tp']}")
    print("=" * 76 + "\n")
    return m_opt

def plot_evaluation_charts(target_name, y_te, test_probs, optimal_tau):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fpr, tpr, _ = roc_curve(y_te, test_probs)
    prec_pts, rec_pts, _ = precision_recall_curve(y_te, test_probs)
    auc_val = roc_auc_score(y_te, test_probs)
    auprc_val = average_precision_score(y_te, test_probs)
    
    axes[0].plot(fpr, tpr, label=f"ROC (AUC = {auc_val:.3f})", color="darkorange", lw=2)
    axes[0].plot(rec_pts, prec_pts, label=f"PR (AUC = {auprc_val:.3f})", color="purple", lw=2)
    axes[0].plot([0, 1], [0, 1], color="gray", linestyle=":")
    axes[0].set_title(f"ROC & PR Curves: {target_name}")
    axes[0].set_xlabel("FPR / Recall")
    axes[0].set_ylabel("TPR / Precision")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    cm_d1 = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_te, (test_probs >= 0.50).astype(int)), display_labels=["Neg", "Pos"])
    cm_d1.plot(ax=axes[1], cmap="Reds", colorbar=False)
    axes[1].set_title("CM at tau=0.50")

    cm_d2 = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_te, (test_probs >= optimal_tau).astype(int)), display_labels=["Neg", "Pos"])
    cm_d2.plot(ax=axes[2], cmap="Greens", colorbar=False)
    axes[2].set_title(f"CM at Optimal tau*={optimal_tau:.2f}")

    plt.tight_layout()
    plt.show()

def save_scaler_params_to_json(scaler, feature_names_list, json_path):
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    scaler_dict = OrderedDict([
        ("mean", OrderedDict((feat, float(scaler.mean_[i])) for i, feat in enumerate(feature_names_list))),
        ("variance", OrderedDict((feat, float(scaler.var_[i])) for i, feat in enumerate(feature_names_list))),
        ("std", OrderedDict((feat, float(scaler.scale_[i])) for i, feat in enumerate(feature_names_list)))
    ])
    with open(json_path, "w") as f:
        json.dump(scaler_dict, f, indent=4)
    print(f"[Scaler Export] Saved Scaler parameters to JSON: {json_path}")
    return scaler_dict

def get_or_fit_scaler(json_path, feature_names_list, X_train=None, force_recompute=False):
    scaler = StandardScaler()
    if not force_recompute and os.path.exists(json_path):
        try:
            with open(json_path, "r") as f:
                sc_data = json.load(f)
            if "mean" in sc_data and "std" in sc_data:
                mean_vals = [sc_data["mean"][k] for k in feature_names_list if k in sc_data["mean"]]
                scale_vals = [sc_data["std"][k] for k in feature_names_list if k in sc_data["std"]]
                var_vals = [sc_data.get("variance", {}).get(k, sc_data["std"][k]**2) for k in feature_names_list if k in sc_data.get("variance", sc_data["std"])]
                if len(mean_vals) == len(feature_names_list) and len(scale_vals) == len(feature_names_list):
                    scaler.mean_ = np.array(mean_vals, dtype=np.float64)
                    scaler.scale_ = np.array(scale_vals, dtype=np.float64)
                    scaler.var_ = np.array(var_vals, dtype=np.float64)
                    scaler.n_features_in_ = len(feature_names_list)
                    print(f"[Scaler Cache Hit] Loaded existing StandardScaler from: {json_path}")
                    return scaler
        except Exception as e:
            print(f"[Scaler Warning] Failed to load {json_path} ({e}). Refitting...")
            
    print(f"[Scaler Compute] Fitting StandardScaler from training data...")
    if X_train is not None and len(X_train) > 0:
        scaler.fit(X_train)
        save_scaler_params_to_json(scaler, feature_names_list, json_path)
    return scaler


In [ ]:
# ==============================================================================
# Target: Future_Hypotension
# ==============================================================================
target_name = "Future_Hypotension"
print(f"=== [TRAINING] Logistic Regression for {target_name} ===")

X_tr, y_tr = build_dataset_matrix_19(train_subset, target_name, oversample_factor=2)
X_va, y_va = build_dataset_matrix_19(val_subset, target_name, oversample_factor=1)
X_te, y_te = build_dataset_matrix_19(test_subset, target_name, oversample_factor=1)

scaler_path = os.path.join("models", "scalers", f"scaler_{target_name}.json")
scaler = get_or_fit_scaler(scaler_path, features_19, X_tr, force_recompute=FORCE_RECOMPUTE_SCALER)

X_tr_sc = scaler.transform(X_tr)
X_va_sc = scaler.transform(X_va)
X_te_sc = scaler.transform(X_te)

from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(class_weight="balanced", max_iter=400, C=1.0, random_state=42)
lr.fit(X_tr_sc, y_tr)

val_probs = lr.predict_proba(X_va_sc)[:, 1]
tau_lr = get_optimal_tau(y_va, val_probs)
test_probs = lr.predict_proba(X_te_sc)[:, 1]
print_metrics_table(target_name, "Logistic Regression (19 feats)", y_te, test_probs, tau_lr)

lr_path = os.path.join("models", "logistic_regression", f"lr_{target_name}.joblib")
os.makedirs(os.path.dirname(lr_path), exist_ok=True)
joblib.dump(lr, lr_path)
print(f"Saved Logistic Regression model to: {lr_path}")

plot_evaluation_charts(f"{target_name} (Logistic Regression 19 Feats)", y_te, test_probs, tau_lr)

del X_tr, y_tr, X_va, y_va, X_te, y_te, X_tr_sc, X_va_sc, X_te_sc
gc.collect()


In [ ]:
# ==============================================================================
# Target: Future_Hypoxia
# ==============================================================================
target_name = "Future_Hypoxia"
print(f"=== [TRAINING] Logistic Regression for {target_name} ===")

X_tr, y_tr = build_dataset_matrix_19(train_subset, target_name, oversample_factor=3)
X_va, y_va = build_dataset_matrix_19(val_subset, target_name, oversample_factor=1)
X_te, y_te = build_dataset_matrix_19(test_subset, target_name, oversample_factor=1)

scaler_path = os.path.join("models", "scalers", f"scaler_{target_name}.json")
scaler = get_or_fit_scaler(scaler_path, features_19, X_tr, force_recompute=FORCE_RECOMPUTE_SCALER)

X_tr_sc = scaler.transform(X_tr)
X_va_sc = scaler.transform(X_va)
X_te_sc = scaler.transform(X_te)

from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(class_weight="balanced", max_iter=400, C=1.0, random_state=42)
lr.fit(X_tr_sc, y_tr)

val_probs = lr.predict_proba(X_va_sc)[:, 1]
tau_lr = get_optimal_tau(y_va, val_probs)
test_probs = lr.predict_proba(X_te_sc)[:, 1]
print_metrics_table(target_name, "Logistic Regression (19 feats)", y_te, test_probs, tau_lr)

lr_path = os.path.join("models", "logistic_regression", f"lr_{target_name}.joblib")
os.makedirs(os.path.dirname(lr_path), exist_ok=True)
joblib.dump(lr, lr_path)
print(f"Saved Logistic Regression model to: {lr_path}")

plot_evaluation_charts(f"{target_name} (Logistic Regression 19 Feats)", y_te, test_probs, tau_lr)

del X_tr, y_tr, X_va, y_va, X_te, y_te, X_tr_sc, X_va_sc, X_te_sc
gc.collect()


In [ ]:
# ==============================================================================
# Target: Future_Tachycardia
# ==============================================================================
target_name = "Future_Tachycardia"
print(f"=== [TRAINING] Logistic Regression for {target_name} ===")

X_tr, y_tr = build_dataset_matrix_19(train_subset, target_name, oversample_factor=2)
X_va, y_va = build_dataset_matrix_19(val_subset, target_name, oversample_factor=1)
X_te, y_te = build_dataset_matrix_19(test_subset, target_name, oversample_factor=1)

scaler_path = os.path.join("models", "scalers", f"scaler_{target_name}.json")
scaler = get_or_fit_scaler(scaler_path, features_19, X_tr, force_recompute=FORCE_RECOMPUTE_SCALER)

X_tr_sc = scaler.transform(X_tr)
X_va_sc = scaler.transform(X_va)
X_te_sc = scaler.transform(X_te)

from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(class_weight="balanced", max_iter=400, C=1.0, random_state=42)
lr.fit(X_tr_sc, y_tr)

val_probs = lr.predict_proba(X_va_sc)[:, 1]
tau_lr = get_optimal_tau(y_va, val_probs)
test_probs = lr.predict_proba(X_te_sc)[:, 1]
print_metrics_table(target_name, "Logistic Regression (19 feats)", y_te, test_probs, tau_lr)

lr_path = os.path.join("models", "logistic_regression", f"lr_{target_name}.joblib")
os.makedirs(os.path.dirname(lr_path), exist_ok=True)
joblib.dump(lr, lr_path)
print(f"Saved Logistic Regression model to: {lr_path}")

plot_evaluation_charts(f"{target_name} (Logistic Regression 19 Feats)", y_te, test_probs, tau_lr)

del X_tr, y_tr, X_va, y_va, X_te, y_te, X_tr_sc, X_va_sc, X_te_sc
gc.collect()
